In [31]:
import os
import pandas as pd
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter

load_dotenv()

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [4]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and ROOT.parent.name == "src":
    ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

#BASE_DIR = Path(__file__).resolve().parents[2]
DATA_PATH = ROOT / "data" / "evenements-publics-openagenda.csv"

In [5]:
raw = pd.read_csv(DATA_PATH, sep=";")
raw.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,25926303,venez-rencontrer-yes-industrie-et-logistique-p...,https://openagenda.com/francetravail/events/ve...,Venez rencontrer YES (industrie et logistique)...,Présence de l'agence d'emploi YES de 11h à 12h...,<p>Présence de l'agence d'emploi YES de 11h à ...,NaN,"INTERIM,#TousMobilisés,Marché du travail,Recru...",https://cdn.openagenda.com/main/05c4bea3d69946...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
1,9825858,atelier-decouverte-du-numerique-8424261,https://openagenda.com/francetravail/events/at...,ATELIER DECOUVERTE DU NUMERIQUE,Vous souhaitez vous orienter et travailler dan...,<p>Vous souhaitez vous orienter et travailler ...,NaN,"L alternance,Métiers du numérique et informati...",https://cdn.openagenda.com/main/0ef2e72ec05543...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
2,30636112,rencontrez-synergie-proxi-latecoere-logistique...,https://openagenda.com/francetravail/events/re...,Rencontrez SYNERGIE PROXI LATECOERE (logistiqu...,Présence de l'agence Synergie Proxi Latécoère ...,<p>Présence de l'agence Synergie Proxi Latécoè...,NaN,"1 jeune 1 solution,#TousMobilisés,Recrutement",https://cdn.openagenda.com/main/f710e417553841...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
3,38746001,handicap-et-emploi-les-cles-de-ma-reussite-res...,https://openagenda.com/francetravail/events/ha...,Handicap et emploi : les clés de ma réussite (...,Objectifs : vous apporter les éléments clés qu...,<p>Objectifs : vous apporter les éléments clés...,NaN,"Se préparer,Marché du travail",https://cdn.openagenda.com/main/7faa84049d9640...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
4,85266313,eiffage-energie-systemes-formation-metier-mont...,https://openagenda.com/francetravail/events/ei...,EIFFAGE ENERGIE SYSTEMES : Formation métier Mo...,Vous cherchez à vous orienter vers un métier q...,<p>Vous cherchez à vous orienter vers un métie...,NaN,"#TousMobilisés,Recrutement,Découverte formatio...",https://cdn.openagenda.com/main/d097609d3acd4f...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN


In [9]:
raw.description_fr.describe()

count                                                  4904
unique                                                 3048
top       Le Job Dating des Minimes se déroule le mardi ...
freq                                                     93
Name: description_fr, dtype: object

Il y n'y a que 3048 valeurs uniques dans la colonne descrition_fr du tableau de donnée. Traiter les mêmes éléments avec le modèle d'embedding textuel créerai les mêmes embeddings, on aurait des valeurs en doubles dans la base de données, ce n'est pas utile. Je ne vais donc conserver que les valeurs uniques pour la création de la table d'embeddings.

In [21]:
unique_description_fr = raw.description_fr.unique()
unique_description_fr.isna().sum()

np.int64(1)

unique_description_fr contient une valeur nulle qu'on va pouvoir supprimer avec la fonction dropna

In [22]:
unique_description_fr = unique_description_fr.dropna()

In [25]:
def get_embeddings(texts, model="text-embedding-3-small"):
    client = OpenAI()
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    return response.data[0].embedding

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialisation du splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Taille de chaque segment
    chunk_overlap=200 # Chevauchement entre les segments
    )
    
# Découpage du texte
segments = text_splitter.split_text("Votre texte ici")

In [26]:
embeddings_list = []
for text in unique_description_fr:
    embedding = get_embeddings(text)
    embeddings_list.append(embedding)

print(f"Number of embeddings: {len(embeddings_list)}")

Number of embeddings: 3048


In [24]:
print(response.data[0].embedding)

NameError: name 'response' is not defined

In [7]:


# 1. Cette ligne lit le fichier .env et charge la clé en mémoire
load_dotenv()

# 2. Le client OpenAI cherche automatiquement la variable "OPENAI_API_KEY"
client = OpenAI()

# 3. Vous pouvez maintenant utiliser l'API normalement
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "Explique-moi la lune en une phrase."}
    ]
)

print(completion.choices[0].message.content)


La Lune est le satellite naturel de la Terre, éclairée par le Soleil, influençant les marées et fascinant l'humanité par ses phases changeantes et son impact culturel et scientifique.


In [6]:
print(completion.choices[0].message.content)

La lune est le satellite naturel de la Terre, illuminée en partie par le soleil, et influençant les marées terrestres.


In [8]:
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here", model="text-embedding-3-small"
)

print(response.data[0].embedding)

[0.005115509033203125, 0.017242431640625, -0.018768310546875, -0.0185699462890625, -0.047271728515625, -0.030303955078125, 0.027679443359375, 0.0036468505859375, 0.01125335693359375, 0.006404876708984375, -0.001678466796875, 0.0158233642578125, -0.001255035400390625, -0.007843017578125, 0.0599365234375, 0.050323486328125, -0.0275421142578125, 0.00994110107421875, -0.04034423828125, 0.050018310546875, -0.00043082237243652344, 0.030181884765625, -0.0137176513671875, 0.032958984375, 0.0173187255859375, 0.0167999267578125, -0.0017566680908203125, 0.0204620361328125, 0.040802001953125, -0.0377197265625, -0.026153564453125, -0.050048828125, 0.024200439453125, -0.05523681640625, -0.0322265625, 0.0423583984375, 0.064697265625, 0.01473236083984375, -0.01568603515625, -0.041351318359375, 0.02215576171875, 0.007411956787109375, 0.044952392578125, 0.0070953369140625, -0.0240936279296875, 0.05242919921875, -0.0200958251953125, -0.0322265625, 0.0162811279296875, 0.046478271484375, 0.0238494873046875